# First Implementation of the RSA model 

In [11]:
import numpy as np 
import random as rd
import spacy
import random as rd
import json 
import sys 
#sys.path.insert(1, '/home/user/Documents/Sprachgebrauchsmodellierung_RSA/Hausarbeit_RSA/repo/scripts/')
#sys.path.insert(1, 'D:StandardLibs/documents/Uni/Sprachgebrauchsmodellierung_RSA/Hausarbeit/imposter_RSA/scripts/')
from gamestate import gs
nlp = spacy.load('de_core_news_sm')


Vocabulary loaded from D:/StandardLibs/documents/Uni/Sprachgebrauchsmodellierung_RSA/Hausarbeit/imposter_RSA/output/random_nouns.json
randomly chosen target word: Fussballer
randomly chosen play order: ['P2', 'P3', 'P1']
similarity table loaded from D:/StandardLibs/documents/Uni/Sprachgebrauchsmodellierung_RSA/Hausarbeit/imposter_RSA/output/sims.npy


In [13]:
logger = logging.getLogger(__name__)
logging.basicConfig(filename='../output/logs/probs.log', level=logging.INFO)

In [14]:
print(gs.V)

['Notwendigkeit', 'Zugang', 'Glanz', 'Band', 'Online', 'Fussballer', 'Filter', 'Erholung', 'U00', 'Produkt']


In [4]:
class Agent:
    def __init__(self, player, knows_imp=False): 
        self.player = player
        self.role = gs.ROLES[player]
        self.alpha = gs.RATIONALITY[player]
        if knows_imp: 
            self.priors = {}
            for p in gs.PLAYERS:
                if p==self.player:
                    self.priors[p]= self.get_priors(p)
                elif gs.ROLES[p]=="agent":
                    self.priors[p]= self.get_priors(p, knows_imp=False)
                else:
                    self.priors[p]= self.get_priors(p, knows_t=False)
        self.target_imp = self.get_target()
        


    def get_target(self):
        #max_freq = max(gs.FREQ.values())
        imp = [p for p in gs.PLAYERS if gs.ROLES[p]=="imposter"][0]
        t_ind = np.argmax(self.priors[imp][gs.PLAYERS.index(imp)])
        #res = [k for k in gs.FREQ if gs.FREQ[k] == max_freq]
        return(gs.V[t_ind])
    
    def get_priors(self, player, knows_t = True, knows_imp = True):
        priors = []
        # P1 weiß wer der Imposter ist 
        for p, w in gs.W: 
            if knows_imp: 
                if gs.ROLES[p]=="agent":
                    priors.append(0)
                else:
                    if knows_t: 
                        if w==gs.TARGET:
                            priors.append(1)
                        else:
                            priors.append(0)
                    else:
                        priors.append(gs.FREQ[w])
            else:
                if p==player:
                    priors.append(0)
                else:
                    if w==gs.TARGET:
                        priors.append(1/(len(gs.PLAYERS)-1))
                    else:
                        priors.append(0)
                    #priors.append(1/(len(gs.PLAYERS)-1)
                
                    
        priors = [w/sum(priors) for w in priors]
        priors = np.array(priors)
        #print(f"priors shape: {priors.shape}, players: {gs.PLAYERS}, V: {gs.V}")
        priors.shape = (len(gs.PLAYERS),len(gs.V))
        return(priors)
    
    def get_P_u_w(self,u,w,t_imp,speaker="P1"):
        imp_ind, t_ind = w
        # sollte so passen, weil wenn L=imp sind Berechnungen egal, in denen nicht S = imp
        if gs.PLAYERS[imp_ind]==speaker: 
            P_u_w = gs.SIMS[t_imp, u]
        else:
            P_u_w = gs.SIMS[t_ind, u]

        return(P_u_w)

    def L0(self, u, w, listener="P2", speaker="P1"):
        logger.info(f"calculating P({w}|{u}) for L0...")
        imp, t = w
        imp = gs.PLAYERS.index(imp)
        t = gs.V.index(t)
        u = gs.V.index(u)
        t_imp = gs.V.index(self.target_imp)
        priors_l = self.priors[listener]

        logger.info(f"P(u|w) in L(0): {self.get_P_u_w(u,(imp,t),t_imp) }, priors: {priors_l[imp][t]}")
        P_w_u = self.get_P_u_w(u,(imp,t),t_imp, speaker=speaker) * priors_l[imp][t]
        logger.info(f"P(w|u) in L(0): {P_w_u}")
        SUM = np.nansum([self.get_P_u_w(u,(gs.PLAYERS.index(w_n[0]), gs.V.index(w_n[1])),t_imp, speaker=speaker) * priors_l[gs.PLAYERS.index(w_n[0]),gs.V.index(w_n[1])] for w_n in gs.W])
        logger.info(f"SUM in L0: {SUM}")
        if SUM==0:
            P_w_u = 0
        else:    
            P_w_u = P_w_u/SUM
            
        logger.info(f"normalized P(w|u) in L(0): {P_w_u}")

        return(P_w_u)


    
    def utility(self, u, w, partner="P2", speaker="P1", imp="P3"):
        #print(f"listen(u,w,listener=partner): {listen(u,w,listener=partner)}, listen(u,(P3,t),listener=imp): {listen(u,(imp,w[1]),listener=imp)}")
        # Anwendung von log-Funktion auf Differenz beruht ausschließlich auf ausprobieren
        #print(f"partner part of utility:\n with logs: {np.log(self.L0(u,w,listener=partner))}, without logs:{self.L0(u,w,listener=partner)}\n imposter part of utility:\n with logs: {np.log(self.L0(u,(imp,w[1]),listener=imp))}, without logs:{self.L0(u,(imp,w[1]),listener=imp)}\n")
        U = np.log(self.L0(u,w,listener=partner, speaker=speaker)) + np.log(self.L0(u,(imp,w[1]),listener=imp, speaker=speaker)) # weil auf logarithmischer Skala negativ 
        #SUM = np.nansum([np.log(listen(u_n,w,listener=partner)) - np.log(listen(u_n,w,listener=imp)) for u_n in range(len(V))])
        logger.info(f"U({u},{w}): {U}")
        return(U)
    
    def S1(self, u,w, alpha=1, speaker="P1"): 
        P_u_w = np.exp(alpha * self.utility(u,w, speaker=speaker))
        logger.info(f"P({u}|{w}) in S1: {P_u_w}")
        SUM = np.nansum([np.exp(alpha * self.utility(u_n, w, speaker=speaker)) for u_n in gs.V])
        logger.info(f"sum of P({u}|w) for each u E V: {SUM}")
        P_u_w = P_u_w/SUM
        logger.info(f"normalized P({u}|{w}) in S1: {P_u_w}")
        return(P_u_w)
    
    def L1(self, u,w,alpha=1,speaker="P2", listener="P1"):
        imp, t = w
        imp_ind = gs.PLAYERS.index(imp)
        t_ind = gs.V.index(t)
        # WHAT IS THE DIFFERENCE
        if imp==speaker:
            P_w_u = self.S1(u,w,alpha=alpha) * self.priors[listener][imp_ind][t_ind]
        else:
            #print(f"P(u|w): {self.S1(u,w,alpha=alpha)}, prior: {self.priors[listener][imp_ind][t_ind]}")
            P_w_u = self.S1(u,w,alpha=alpha) * self.priors[listener][imp_ind][t_ind]
        #print("P(w|u) before normalization: ", P_w_u)
        SUM = np.nansum([self.S1(u,w_n,alpha=alpha) * self.priors[listener][gs.PLAYERS.index(w_n[0])][gs.V.index(w_n[1])] for w_n in gs.W])
        #print("probability sum: ", SUM)

        P_w_u = P_w_u/SUM
        return(P_w_u)

    def collect_l_posteriors(self, listener, u, speaker="P1", l_level = 0): 
        posteriors = []
        for w in gs.W: 
            if l_level==0: 
                posterior = self.L0(u, w, speaker=speaker, listener=listener)
            else: 
                posterior = self.L1(u, w, alpha=gs.RATIONALITY[listener], speaker=speaker, listener=listener)
            posteriors.append(posterior)
        
        posteriors = np.array(posteriors)
        posteriors.shape = (len(gs.PLAYERS),len(gs.V))
        return(posteriors)


    def adjust_priors(self, player, probs): 
        adjusted_priors = (self.priors[player] + probs)/2
        self.priors[player] = adjusted_priors

    def listen(self, u, speaker):
        for p in gs.PLAYERS:
            if not p==speaker:
                if p==self.player:
                    posteriors = self.collect_l_posteriors(p, u, speaker=speaker, l_level= 1)
                else:
                    posteriors = self.collect_l_posteriors(p, u, speaker=speaker, l_level= 0)
                print(f"posteriors {p}: {posteriors}")
                self.adjust_priors(p, posteriors)
        self.target_imp = self.get_target()



In [25]:
class Agent:
    def __init__(self, player, knows_imp=False): 
        self.player = player
        self.role = gs.ROLES[player]
        self.alpha = gs.RATIONALITY[player]
        self.priors = self.get_priors_per_player()
        self.targets_imp = {p: self.get_target(p) for p in gs.PLAYERS if p!=self.player}

    def get_priors_per_player(self): 
        priors = {p: {} for p in gs.PLAYERS}
        for p in gs.PLAYERS: 
            for role in ["imposter", "agent"]: 
                if role=="imposter":
                    ki = True
                    kt = False
                else:
                    ki = False
                    kt = True

                i_priors, t_priors = self.get_priors(p, knows_t=kt, knows_imp=ki)
                priors[p][role]={}
                priors[p][role]["i"]=i_priors
                priors[p][role]["t"]=t_priors
        return(priors)

    def get_target(self, player):
        #max_freq = max(gs.FREQ.values())
        #imp = [p for p in gs.PLAYERS if gs.ROLES[p]=="imposter"][0]
        t_ind = np.argmax(self.priors[player]["imposter"]["t"])
        #res = [k for k in gs.FREQ if gs.FREQ[k] == max_freq]
        return(gs.V[t_ind])


    def get_priors(self, player, knows_t = True, knows_imp = True):
        i_priors = []
        t_priors = []
        # P1 weiß nicht wer der Imposter ist 
        for p in gs.PLAYERS: 
            if knows_imp: 
                if gs.ROLES[p]=="agent":
                    i_priors.append(0)
                else:
                    i_priors.append(1)
            else:
                if p==player:
                    i_priors.append(0)
                else: 
                    p_num = len(gs.PLAYERS)
                    i_priors.append(1/(p_num-1))
        
        for w in gs.V: 
            if knows_t: 
                if w==gs.TARGET:
                    t_priors.append(1)
                else:
                    t_priors.append(0)
            else:
                t_priors.append(gs.FREQ_NORMED[w])
                                
                    
        i_priors = np.array(i_priors)
        t_priors = np.array(t_priors)

        return((i_priors, t_priors))
    
    def get_P_u_w(self,u,w,t_imp,speaker="P1"):
        imp_ind, t_ind = w
        # sollte so passen, weil wenn L=imp sind Berechnungen egal, in denen nicht S = imp
        # kommt mir sehr sus vor - warum wird einmal die ähnlichkeit mit dem in der welt enthalteten target berechnet und einmal mit dem target von imp?
        if gs.PLAYERS[imp_ind]==speaker: 
            P_u_w = gs.SIMS[t_imp, u]
        else:
            P_u_w = gs.SIMS[t_ind, u]
            
        return(P_u_w)

    def L0(self, u, w, listener="P2", speaker="P1"):
        logger.info(f"calculating P({w}|{u}) for L0...")
        imp, t = w
        imp = gs.PLAYERS.index(imp)
        t = gs.V.index(t)
        u = gs.V.index(u)
        t_imp = gs.V.index(self.target_imp)
        priors_l = self.priors[listener]

        logger.info(f"P(u|w) in L(0): {self.get_P_u_w(u,(imp,t),t_imp) }, priors: {priors_l[imp][t]}")
        P_w_u = self.get_P_u_w(u,(imp,t),t_imp, speaker=speaker) * priors_l[imp][t]
        logger.info(f"P(w|u) in L(0): {P_w_u}")
        SUM = np.nansum([self.get_P_u_w(u,(gs.PLAYERS.index(w_n[0]), gs.V.index(w_n[1])),t_imp, speaker=speaker) * priors_l[gs.PLAYERS.index(w_n[0]),gs.V.index(w_n[1])] for w_n in gs.W])
        logger.info(f"SUM in L0: {SUM}")
        if SUM==0:
            P_w_u = 0
        else:    
            P_w_u = P_w_u/SUM
            
        logger.info(f"normalized P(w|u) in L(0): {P_w_u}")

        return(P_w_u)
    
    def get_P_u_i(self,u,imp,t_imp,speaker="P1"):
        # sollte so passen, weil wenn L=imp sind Berechnungen egal, in denen nicht S = imp
        # kommt mir sehr sus vor - warum wird einmal die ähnlichkeit mit dem in der welt enthalteten target berechnet und einmal mit dem target von imp?
        if gs.PLAYERS[imp]==speaker: 
            P_u_w = gs.SIMS[t_imp, u]
        else:
            P_u_w = gs.SIMS[gs.V.index(gs.TARGET), u]

        return(P_u_w)
    
    def L0_i(self, u, i, listener="P2", speaker="P1", role="agent"):
        #logger.info(f"calculating P({w}|{u}) for L0...")
        #imp, t = w
        imp = gs.PLAYERS.index(i)
        #t = gs.V.index(t)
        u = gs.V.index(u)
        t_imp = gs.V.index(self.targets_imp[i])
        #priors_l_a = self.priors[listener]["agent"]["i"]
        priors_l = self.priors[listener][role]["i"]

        #logger.info(f"P(u|w) in L(0): {self.get_P_u_w(u,(imp,t),t_imp) }, priors: {priors_l[imp][t]}")
        P_i_u = self.get_P_u_i(u,imp,t_imp,speaker=speaker) * priors_l[imp]
        #logger.info(f"P(w|u) in L(0): {P_w_u}")
        SUM = np.nansum([self.get_P_u_i(u,i_n,t_imp, speaker=speaker) * priors_l[i_n] for i_n in range(len(gs.PLAYERS))])
        #logger.info(f"SUM in L0: {SUM}")
        if SUM==0:
            P_i_u = 0
        else:    
            P_i_u = P_i_u/SUM
            
        #logger.info(f"normalized P(w|u) in L(0): {P_w_u}")

        return(P_i_u)
    
    def get_P_u_t(self,u,t): 

        P_u_t = gs.SIMS[t, u]

        return(P_u_t)

    
    def L0_t(self, u, t, listener="P3"): 
        t = gs.V.index(t)
        u = gs.V.index(u)
        priors_l = self.priors[listener]
        P_t_u = self.get_P_u_t(u,t) * priors_l["imposter"]["t"][t]
        #print("divisor list: ", [self.get_P_u_t(u,t) * priors_l["imposter"]["t"][t_n] for t_n in range(gs.V)])
        SUM = np.nansum([self.get_P_u_t(u,t_n) * priors_l["imposter"]["t"][t_n] for t_n in range(len(gs.V))])
        if SUM==0:
            P_t_u = 0
        else:    
            P_t_u = P_t_u/SUM
        return(P_t_u)
    
    def utility(self, u, i, t, partner="P2", speaker="P1", x=5):
        #print(f"listen(u,w,listener=partner): {listen(u,w,listener=partner)}, listen(u,(P3,t),listener=imp): {listen(u,(imp,w[1]),listener=imp)}")
        # Anwendung von log-Funktion auf Differenz beruht ausschließlich auf ausprobieren
        #print(f"partner part of utility:\n with logs: {np.log(self.L0(u,w,listener=partner))}, without logs:{self.L0(u,w,listener=partner)}\n imposter part of utility:\n with logs: {np.log(self.L0(u,(imp,w[1]),listener=imp))}, without logs:{self.L0(u,(imp,w[1]),listener=imp)}\n")
        #print(f"utterance: {u}, imposter: {i}")
        #print(f"x: {x}\nl0_t: {self.L0_t(u,t,listener=imp)}\nl0_i: {self.L0_i(u,i,listener=partner, speaker=speaker, role="agent")}")
        U = np.log(x * self.L0_t(u,t,listener=i)) - np.log(self.L0_i(u,i,listener=partner, speaker=speaker, role="agent"))
        #print("U: ", U)
        #U = np.log(self.L0(u,gs.V.index(gs.TARGET),listener=partner, speaker=speaker)) + np.log(self.L0(u,(imp,w[1]),listener=imp, speaker=speaker)) # weil auf logarithmischer Skala negativ 
        #SUM = np.nansum([np.log(listen(u_n,w,listener=partner)) - np.log(listen(u_n,w,listener=imp)) for u_n in range(len(V))])
        #logger.info(f"U({u},{w}): {U}")
        return(U)
    
    def S1_i_t(self, u, i, t, alpha=1, speaker="P1"):
        # gets other player that is not imposter
        partner = [p for p in gs.PLAYERS if not p in [speaker,i]][0]
        P_u = np.exp(alpha*self.utility(u,i,t, partner=partner, speaker=speaker))
        SUM = np.nansum([np.exp(alpha*self.utility(u_n, i, t, partner=partner, speaker=speaker)) for u_n in gs.V])
        if SUM==0:
            P_u = 0
        else:
            P_u = P_u/SUM

        return(P_u)
    
    def S1_certain(self, u, t, speaker="P1", alpha=1): 
        
        def get_i_sum(u,t,speaker="P1", alpha=1):
            P_u = sum([self.priors[speaker]["agent"]["i"][gs.PLAYERS.index(i)] * self.S1_i_t(u,i,t, alpha=alpha, speaker=speaker) for i in gs.PLAYERS if i!=self.player])
            return(P_u)

        P_u = get_i_sum(u,t,speaker=speaker, alpha=alpha)
        SUM = np.nansum([get_i_sum(u_n,t,speaker=speaker, alpha=alpha) for u_n in gs.V])
        if SUM == 0: 
            P_u = 0 
        else:
            P_u = P_u/SUM
        return(P_u)
        
    
    
    def L1(self, u,w,alpha=1,speaker="P2", listener="P1"):
        imp, t = w
        imp_ind = gs.PLAYERS.index(imp)
        t_ind = gs.V.index(t)
        # WHAT IS THE DIFFERENCE
        if imp==speaker:
            P_w_u = self.S1(u,w,alpha=alpha) * self.priors[listener][imp_ind][t_ind]
        else:
            #print(f"P(u|w): {self.S1(u,w,alpha=alpha)}, prior: {self.priors[listener][imp_ind][t_ind]}")
            P_w_u = self.S1(u,w,alpha=alpha) * self.priors[listener][imp_ind][t_ind]
        #print("P(w|u) before normalization: ", P_w_u)
        SUM = np.nansum([self.S1(u,w_n,alpha=alpha) * self.priors[listener][gs.PLAYERS.index(w_n[0])][gs.V.index(w_n[1])] for w_n in gs.W])
        #print("probability sum: ", SUM)

        P_w_u = P_w_u/SUM
        return(P_w_u)

    def collect_l_posteriors(self, listener, u, speaker="P1", l_level = 0): 
        posteriors = []
        for w in gs.W: 
            if l_level==0: 
                posterior = self.L0(u, w, speaker=speaker, listener=listener)
            else: 
                posterior = self.L1(u, w, alpha=gs.RATIONALITY[listener], speaker=speaker, listener=listener)
            posteriors.append(posterior)
        
        posteriors = np.array(posteriors)
        posteriors.shape = (len(gs.PLAYERS),len(gs.V))
        return(posteriors)


    def adjust_priors(self, player, probs): 
        adjusted_priors = (self.priors[player] + probs)/2
        self.priors[player] = adjusted_priors

    def listen(self, u, speaker):
        for p in gs.PLAYERS:
            if not p==speaker:
                if p==self.player:
                    posteriors = self.collect_l_posteriors(p, u, speaker=speaker, l_level= 1)
                else:
                    posteriors = self.collect_l_posteriors(p, u, speaker=speaker, l_level= 0)
                print(f"posteriors {p}: {posteriors}")
                self.adjust_priors(p, posteriors)
        self.target_imp = self.get_target()



In [56]:
class Speaker_Agent:
    def __init__(self, player, knows_imp=False): 
        self.player = player
        self.role = gs.ROLES[player]
        self.alpha = gs.RATIONALITY[player]
        self.priors = self.get_priors_per_player()

    def get_priors_per_player(self): 
        priors = {p: {} for p in gs.PLAYERS}
        for p in gs.PLAYERS: 

            for role in ["imposter", "agent"]: 
                if role=="imposter":
                    ki = True
                    kt = False
                else:
                    ki = False
                    kt = True

                i_priors, t_priors = self.get_priors(p, knows_t=kt, knows_imp=ki)
                priors[p][role]={}
                priors[p][role]["i"]=i_priors
                priors[p][role]["t"]=t_priors
        return(priors)



    def get_priors(self, player, knows_t = True, knows_imp = True):
        i_priors = []
        t_priors = []
        # P1 weiß nicht wer der Imposter ist 
        for p in gs.PLAYERS: 
            if knows_imp: 
                if gs.ROLES[p]=="agent":
                    i_priors.append(0)
                else:
                    i_priors.append(1)
            else:
                if p==player:
                    i_priors.append(0)
                else: 
                    p_num = len(gs.PLAYERS)
                    i_priors.append(1/(p_num-1))
        
        for w in gs.V: 
            if knows_t: 
                if w==gs.TARGET:
                    t_priors.append(1)
                else:
                    t_priors.append(0)
            else:
                t_priors.append(gs.FREQ_NORMED[w])
                                
                    
        i_priors = np.array(i_priors)
        t_priors = np.array(t_priors)

        return((i_priors, t_priors))
    

    def get_P_u_w(self,u,w,speaker="P1"):
        imp_ind, t_ind = w
        # sollte so passen, weil wenn L=imp sind Berechnungen egal, in denen nicht S = imp
        # kommt mir sehr sus vor - warum wird einmal die ähnlichkeit mit dem in der welt enthalteten target berechnet und einmal mit dem target von imp?
        if gs.PLAYERS[imp_ind]==speaker: 
            P_u_w = sum([self.priors[speaker]["imposter"]["t"][n] * gs.SIMS[n, u] for n in range(len(gs.V))])
        else:
            P_u_w = gs.SIMS[t_ind, u]
            
        return(P_u_w)

    def L0_a(self, u, w, listener="P2", speaker="P1"):
        #logger.info(f"calculating P({w}|{u}) for L0...")
        imp, t = w
        imp = gs.PLAYERS.index(imp)
        t = gs.V.index(t)
        u = gs.V.index(u)
        priors_l = self.priors[listener]

        #logger.info(f"P(u|w) in L(0): {self.get_P_u_w(u,(imp,t),t_imp) }, priors: {priors_l[imp][t]}")
        P_w_u = self.get_P_u_w(u,(imp,t), speaker=speaker) * priors_l["agent"]["i"][imp]
        #logger.info(f"P(w|u) in L(0): {P_w_u}")
        SUM = np.nansum([self.get_P_u_w(u,(i_n, t), speaker=speaker) * priors_l["agent"]["i"][i_n] for i_n in range(len(gs.PLAYERS))])
        #logger.info(f"SUM in L0: {SUM}")
        if SUM==0:
            P_w_u = 0
        else:    
            P_w_u = P_w_u/SUM
            
        #logger.info(f"normalized P(w|u) in L(0): {P_w_u}")

        return(P_w_u)


    
    def get_P_u_t(self,u,t): 

        P_u_t = gs.SIMS[t, u]

        return(P_u_t)

    
    def L0_t(self, u, t, listener="P3"): 
        t = gs.V.index(t)
        u = gs.V.index(u)
        priors_l = self.priors[listener]
        P_t_u = self.get_P_u_t(u,t) * priors_l["imposter"]["t"][t]
        SUM = np.nansum([self.get_P_u_t(u,t_n) * priors_l["imposter"]["t"][t_n] for t_n in range(len(gs.V))])
        if SUM==0:
            P_t_u = 0
        else:    
            P_t_u = P_t_u/SUM
        return(P_t_u)
    
    def utility(self, u, i, t, partner="P2", speaker="P1", x=1, log=False):
        l0_t = self.L0_t(u,t,listener=i)
        l0_i = self.L0_a(u,(i,t),listener=partner, speaker=speaker)
        U_i = np.log(x * l0_t)
        U_a = np.log(l0_i)
        U = U_i - U_a
        if log: 
            logger.info(f"outputs of L0 functions within utility function: L0(t|u): {l0_t}, L0(i|u): {l0_i}")
            logger.info(f"imposter component of utility function: {U_i}\nagent component of utility function: {U_a}")

        return(U)
    
    def S1_i_t(self, u, i, t, alpha=1, speaker="P1"):
        # gets other player that is not imposter
        partner = [p for p in gs.PLAYERS if not p in [speaker,i]][0]
        P_u = np.exp(alpha*self.utility(u,i,t, partner=partner, speaker=speaker, log=True))
        SUM = np.nansum([np.exp(alpha*self.utility(u_n, i, t, partner=partner, speaker=speaker)) for u_n in gs.V])
        if SUM==0:
            P_u = 0
        else:
            P_u = P_u/SUM

        return(P_u)
    
    def S1_certain(self, u, t, speaker="P1", alpha=1): 
        
        def get_i_sum(u,t,speaker="P1", alpha=1, log=False):
            P_u = [self.priors[speaker]["agent"]["i"][gs.PLAYERS.index(i)] * self.S1_i_t(u,i,t, alpha=alpha, speaker=speaker) for i in gs.PLAYERS if i!=self.player]
            if log: 
                for n, p in enumerate([player for player in gs.PLAYERS if player!=self.player]):
                    logger.info(f"S1(u|{p},{gs.TARGET})={P_u[n]}")
            P_u_sum = np.nansum(P_u)
            return(P_u_sum)

        P_u = get_i_sum(u,t,speaker=speaker, alpha=alpha, log=True)
        SUM = np.nansum([get_i_sum(u_n,t,speaker=speaker, alpha=alpha) for u_n in gs.V])
        if SUM == 0: 
            P_u = 0 
        else:
            P_u = P_u/SUM
        return(P_u)
        

In [57]:
a1 = Speaker_Agent("P1")

In [58]:
a1.S1_certain(gs.V[0], gs.TARGET)

C:\Users\Hannes\AppData\Local\Temp\ipykernel_12460\1171928174.py:118: RuntimeWarning: divide by zero encountered in log
  U_i = np.log(x * l0_t)
C:\Users\Hannes\AppData\Local\Temp\ipykernel_12460\1171928174.py:119: RuntimeWarning: divide by zero encountered in log
  U_a = np.log(l0_i)
C:\Users\Hannes\AppData\Local\Temp\ipykernel_12460\1171928174.py:120: RuntimeWarning: invalid value encountered in scalar subtract
  U = U_i - U_a


np.float64(0.07860700122204078)

In [39]:
prob_per_u = [a1.S1_certain(u, gs.TARGET) for u in gs.V]
print(prob_per_u)


C:\Users\Hannes\AppData\Local\Temp\ipykernel_12460\2786098677.py:118: RuntimeWarning: divide by zero encountered in log
  U = np.log(x * self.L0_t(u,t,listener=i)) - np.log(self.L0_a(u,(i,t),listener=partner, speaker=speaker))
C:\Users\Hannes\AppData\Local\Temp\ipykernel_12460\2786098677.py:118: RuntimeWarning: invalid value encountered in scalar subtract
  U = np.log(x * self.L0_t(u,t,listener=i)) - np.log(self.L0_a(u,(i,t),listener=partner, speaker=speaker))
C:\Users\Hannes\AppData\Local\Temp\ipykernel_12460\2786098677.py:120: RuntimeWarning: divide by zero encountered in log
  logger.info(f"imposter component of utility function: {np.log(x * self.L0_t(u,t,listener=i))}\nagent component of utility function: {-np.log(self.L0_a(u,(i,t),listener=partner, speaker=speaker))}")


[np.float64(0.07860700122204078), np.float64(0.10257612379252906), np.float64(0.09518372649522834), np.float64(0.11957786619919859), np.float64(0.07989406039142972), np.float64(0.0), np.float64(0.20957542000953439), np.float64(0.15618609592391947), np.float64(0.06950362156707365), np.float64(0.08889608439904603)]


In [34]:
print(gs.SIMS[5])

[0.23703232 0.36651731 0.41496426 0.51218992 0.21370074 0.
 0.99999988 0.8254742  0.12148467 0.18236418 0.19101942]


In [32]:
print(gs.V.index(gs.TARGET))

5


In [ ]:
a1 = Agent("P1")

In [15]:
print(a1.priors)

{'P1': {'agent': {'i': array([0. , 0.5, 0.5]), 't': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0])}}, 'P2': {'imposter': {'i': array([0, 0, 1]), 't': array([0.02902473, 0.07805299, 0.0196113 , 0.22553   , 0.47067129,
       0.0138848 , 0.0241219 , 0.01829735, 0.05275441, 0.06805122])}, 'agent': {'i': array([0.5, 0. , 0.5]), 't': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0])}}, 'P3': {'imposter': {'i': array([0, 0, 1]), 't': array([0.02902473, 0.07805299, 0.0196113 , 0.22553   , 0.47067129,
       0.0138848 , 0.0241219 , 0.01829735, 0.05275441, 0.06805122])}, 'agent': {'i': array([0.5, 0.5, 0. ]), 't': array([0, 0, 0, 0, 0, 0, 1, 0, 0, 0])}}}


In [16]:
print(a1.targets_imp)

{'P2': 'Online', 'P3': 'Online'}


In [17]:
print(gs.V)

['Notwendigkeit', 'Zugang', 'Glanz', 'Band', 'Online', 'Fussballer', 'Filter', 'Erholung', 'U00', 'Produkt']


In [11]:
print(a1.S1_certain("Glanz",gs.TARGET))

utterance: Glanz, imposter: P2
x: 5
l0_t: 0.028088129337034272
l0_i: 0.5814059684003207
U:  -1.4206642976410255
utterance: Notwendigkeit, imposter: P2
x: 5
l0_t: 0.02340952655135881
l0_i: 0.5128091937786018
U:  -1.4773228645038459
utterance: Zugang, imposter: P2
x: 5
l0_t: 0.02862558268711568
l0_i: 0.5869106003173427
U:  -1.4111337789572613
utterance: Glanz, imposter: P2
x: 5
l0_t: 0.028088129337034272
l0_i: 0.5814059684003207
U:  -1.4206642976410255
utterance: Band, imposter: P2
x: 5
l0_t: 0.039625822699804934
l0_i: 0.5727179436298019
U:  -1.0614744439920396
utterance: Online, imposter: P2
x: 5
l0_t: 0.01846034109124766
l0_i: 1.0
U:  -2.3826926603283427
utterance: Fussballer, imposter: P2
x: 5
l0_t: 0.03043759135960442
l0_i: 0.4521971635361458
U:  -1.0890019724171855
utterance: Filter, imposter: P2
x: 5
l0_t: 0.0
l0_i: 0.0
U:  nan
utterance: Erholung, imposter: P2
x: 5
l0_t: 0.06359522902368878
l0_i: 0.7925398570092651
U:  -0.9132664326273886
utterance: U00, imposter: P2
x: 5
l0_t: 0.

/tmp/ipykernel_56655/58310932.py:166: RuntimeWarning: divide by zero encountered in log
  U = np.log(x * self.L0_t(u,t,listener=imp)) - np.log(self.L0_i(u,i,listener=partner, speaker=speaker, role="agent"))
/tmp/ipykernel_56655/58310932.py:166: RuntimeWarning: invalid value encountered in scalar subtract
  U = np.log(x * self.L0_t(u,t,listener=imp)) - np.log(self.L0_i(u,i,listener=partner, speaker=speaker, role="agent"))


In [18]:
for u in gs.V: 
    print(u, a1.S1_certain(u, gs.TARGET))

/tmp/ipykernel_56655/4132279315.py:166: RuntimeWarning: divide by zero encountered in log
  U = np.log(x * self.L0_t(u,t,listener=imp)) - np.log(self.L0_i(u,i,listener=partner, speaker=speaker, role="agent"))
/tmp/ipykernel_56655/4132279315.py:166: RuntimeWarning: invalid value encountered in scalar subtract
  U = np.log(x * self.L0_t(u,t,listener=imp)) - np.log(self.L0_i(u,i,listener=partner, speaker=speaker, role="agent"))


Notwendigkeit 0.0987303738393634
Zugang 0.10548636719276597
Glanz 0.1044858029210875
Band 0.1496412942614111
Online 0.03992580442389372
Fussballer 0.14557821905719098
Filter nan
Erholung 0.17354710654069466
U00 0.08445900239095722
Produkt 0.09814602937263546


In [19]:
for u in gs.V: 
    print(u, a1.L0_i(u, "P3", listener="P2", speaker="P1", role="agent"))


Notwendigkeit 0.5128091937786018
Zugang 0.5869106003173427
Glanz 0.5814059684003207
Band 0.5727179436298019
Online 1.0
Fussballer 0.4521971635361458
Filter 0.0
Erholung 0.7925398570092651
U00 0.5386199945015109
Produkt 0.44339620854005163


In [20]:
for w in gs.V: 
    print(w, a1.L0_t(w, gs.TARGET))

Notwendigkeit 0.02340952655135881
Zugang 0.02862558268711568
Glanz 0.028088129337034272
Band 0.039625822699804934
Online 0.01846034109124766
Fussballer 0.03043759135960442
Filter 0.0
Erholung 0.06359522902368878
U00 0.02103364136231078
Produkt 0.020121055346171063


In [17]:
for u in gs.V: 
    print(u, a1.S1_i_t(u, "P3", gs.TARGET))

Notwendigkeit 0.10929112730416304
Zugang 0.11601944723483582
Glanz 0.10210478561764874
Band nan
Online 0.14475802748668862
Fussballer 0.10887003090067895
Filter 0.11724359739350286
Erholung 0.11088138548380169
U00 0.10165435038108561
Produkt 0.08917724819759464


/tmp/ipykernel_250123/3214175140.py:164: RuntimeWarning: divide by zero encountered in log
  U = np.log(self.L0_t(u,t,listener=imp)) - np.log(self.L0_i(u,i,listener=partner, speaker=speaker, role="agent"))
/tmp/ipykernel_250123/3214175140.py:164: RuntimeWarning: invalid value encountered in scalar subtract
  U = np.log(self.L0_t(u,t,listener=imp)) - np.log(self.L0_i(u,i,listener=partner, speaker=speaker, role="agent"))


In [20]:
for u in range(len(gs.V)): 
    print(gs.V[u], gs.SIMS[gs.V.index(gs.TARGET), u])

Notwendigkeit 0.508949875831604
Zugang 0.50888991355896
Glanz 0.5427209138870239
Band 0.0
Online 1.0
Fussballer 0.4384180009365082
Filter 0.5121899247169495
Erholung 0.5876442790031433
U00 0.3837636709213257
Produkt 0.17019608616828918


In [ ]:
def main(): 
    player_dic = {}
    for p in gs.PLAYERS: 
        role = gs.ROLES[p]
        knows_imp = True if role=="imposter" else False
        player_dic[p] = Agent(p, knows_imp=knows_imp)

        

In [5]:
P1 = Agent("P1", knows_imp=True)


print(gs.TARGET)
print(gs.V.index(gs.TARGET))
print(P1.target_imp)
print(gs.PLAYERS)

Fussballer
5
Online
['P1', 'P2', 'P3']


In [6]:
print(P1.priors)

{'P1': array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]]), 'P2': array([[0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. , 0. ],
       [0. , 0. , 0. , 0. , 0. , 0.5, 0. , 0. , 0. , 0. ]]), 'P3': array([[0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ],
       [0.02902473, 0.07805299, 0.0196113 , 0.22553   , 0.47067129,
        0.0138848 , 0.0241219 , 0.01829735, 0.05275441, 0.06805122]])}


In [7]:
P1.listen("Glanz", "P2")

/tmp/ipykernel_8596/2691900590.py:96: RuntimeWarning: divide by zero encountered in log
  U = np.log(self.L0(u,w,listener=partner, speaker=speaker)) + np.log(self.L0(u,(imp,w[1]),listener=imp, speaker=speaker)) # weil auf logarithmischer Skala negativ
/tmp/ipykernel_8596/2691900590.py:106: RuntimeWarning: invalid value encountered in scalar divide
  P_u_w = P_u_w/SUM


posteriors P1: [[nan nan nan nan nan  0. nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan  1. nan nan nan nan]]
posteriors P3: [[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.03095438 0.17600298 0.         0.27600147 0.39458675 0.01299215
  0.02808813 0.015005   0.02448395 0.04188519]]


In [8]:
P1.target_imp

'Online'

In [9]:
P1.S1(gs.V[0], ("P3", gs.TARGET))

/tmp/ipykernel_8596/2691900590.py:96: RuntimeWarning: divide by zero encountered in log
  U = np.log(self.L0(u,w,listener=partner, speaker=speaker)) + np.log(self.L0(u,(imp,w[1]),listener=imp, speaker=speaker)) # weil auf logarithmischer Skala negativ


np.float64(0.031257468918548595)

In [10]:
P1.target_imp

'Online'

In [11]:
P1.L0("Zugang", ("P3", gs.TARGET), listener="P3", speaker="P2")

np.float64(0.014965368474864337)

In [12]:
P1.L0("Zugang", ("P1", gs.TARGET), listener="P2", speaker="P1")

np.float64(0.4347317054516399)

In [13]:
P1.L1(gs.V[2], ("P3",gs.TARGET), speaker="P1", listener="P2")

/tmp/ipykernel_8596/2691900590.py:96: RuntimeWarning: divide by zero encountered in log
  U = np.log(self.L0(u,w,listener=partner, speaker=speaker)) + np.log(self.L0(u,(imp,w[1]),listener=imp, speaker=speaker)) # weil auf logarithmischer Skala negativ
/tmp/ipykernel_8596/2691900590.py:106: RuntimeWarning: invalid value encountered in scalar divide
  P_u_w = P_u_w/SUM


np.float64(0.3606086300942638)

In [14]:
P1.L0(gs.V[6], ("P3",gs.TARGET))

np.float64(0.8239263146121298)

In [15]:
print(gs.TARGET)
print(P1.target_imp)

Fussballer
Online


In [16]:
for u in gs.V:
    p_w_u = P1.S1(u,("P3",gs.TARGET))
    print(f"u: {u}, P(w|u): {p_w_u}")

/tmp/ipykernel_8596/2691900590.py:96: RuntimeWarning: divide by zero encountered in log
  U = np.log(self.L0(u,w,listener=partner, speaker=speaker)) + np.log(self.L0(u,(imp,w[1]),listener=imp, speaker=speaker)) # weil auf logarithmischer Skala negativ


u: Notwendigkeit, P(w|u): 0.031257468918548595
u: Zugang, P(w|u): 0.08554315717054087
u: Glanz, P(w|u): 0.06117098541898671
u: Band, P(w|u): 0.10689040471083684
u: Online, P(w|u): 0.07658849126323501
u: Fussballer, P(w|u): 0.0
u: Filter, P(w|u): 0.351638658316185
u: Erholung, P(w|u): 0.21629221220461559
u: U00, P(w|u): 0.01920645866754007
u: Produkt, P(w|u): 0.05141216332951143


In [17]:
print(gs.ROLES)

{'P1': 'agent', 'P2': 'agent', 'P3': 'imposter'}


In [18]:
for k,v in P1.priors.items():
    print(f"{k}: {v}")

P1: [[nan nan nan nan nan  0. nan nan nan nan]
 [nan nan nan nan nan nan nan nan nan nan]
 [nan nan nan nan nan  1. nan nan nan nan]]
P2: [[0.  0.  0.  0.  0.  0.5 0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.  0.  0.  0.  0. ]
 [0.  0.  0.  0.  0.  0.5 0.  0.  0.  0. ]]
P3: [[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.02998956 0.12702799 0.00980565 0.25076573 0.43262902 0.01343848
  0.02610502 0.01665117 0.03861918 0.05496821]]


## next steps 

- implement speaker that doesn't know imposter identity
    - do speakers need to explicitely reason about different worlds or is this manageable with the priors only? 
- implement listeners adjusting priors 
- implement imposter 
    - speaking 
    - adjusting priors (about target) 
- implement agents modelling imposter adjusting target probabilities

## questions 

- do (models of) listeners actually have to reason about imposter identity to calculate judgements at all? Am I giving information to the models that they shouldn't actually have? 
- does the utility function actually work the way it's supposed to (penalizing words too close to the target word)?
    - should I add a modifier to the imposter part? 
- 


In [33]:
with open('../output/random_nouns.json', 'r') as file:
    rd_100 = json.load(file)

print(rd_100)


['Ball', 'Künstler', 'Spaziergang', 'Partnerin', 'Bayer', 'Ereignis', 'Feuerwehr', 'Petition', 'Bundestag', 'Erfüllung', 'Ausweis', 'Rückkehr', 'Zitat', 'Verfasst', 'Schülerin', 'Wette', 'Aluminium', 'Saal', 'Einbezog', 'Eign', 'Klinik', 'Playstation', 'Nutze', 'Zufriedenheit', 'Opposition', 'Schweine', 'Messung', 'Steigerung', 'Hafen', 'Service', 'Zwischenzeit', 'Kenntnis', 'Rest', 'Terrorismus', 'Ulm', 'Wettbewerb', 'Freund', 'Element', 'Anwohner', 'Bgb', 'Empfänger', 'Rufe', 'Station', 'Milde', 'Gefasst', 'Invasion', 'Herauszufinde', 'Gewalt', 'Realschule', 'Solidarität', 'Produkt', 'Koffer', 'Mann', 'Verbrechen', 'Weisheit', 'Kontinent', 'Komplex', 'Pressemitteilung', 'Vorteil', 'Doktor', 'Symbol', 'Tasse', 'Scheidung', 'Stürmer', 'Statistik', 'Christ', 'Rundschau', 'Furcht', 'Dialog', 'Partie', 'Nation', 'Haus', 'Jungfrau', 'Tiger', 'Zieh', 'Rufe', 'Hochschule', 'Vorwurf', 'Rechtsprechung', 'Hotel', 'Fahrt', 'Wahrheit', 'Meinung', 'Verkehrt', 'Zwischenzeitlich', 'Einleitung', 'Nac

In [ ]:
class Imposter:
    def __init__(self, player): 
         self.player = player
         self.role = roles[player]
         self.target = self.get_target()
         self.P_t = freq.copy()
         self.alpha = rationality[player]
         self.target_imp = self.get_target()

    def get_target(self):
        mv = max(freq.values())
        res = [k for k in freq if freq[k] == mv]
        return(rd.choice(res))
    
    
    def L0(self, u, t, p_player = [2/3, 1/3]):
        p_agent = get_similarity(u,t) * p_player[0]
        p_imposter = get_similarity(u,self.target_imp) * p_player[1]
        p_agent_norm = p_agent/(p_agent + p_imposter)
        p_imposter_norm = p_imposter/(p_agent + p_imposter)
        return((p_agent_norm, p_imposter_norm))

    def S1(self, t):
        p_distr = []
        for u in V: 
            # what exactly is the world? The persona the player wants to communicate? That would always be agent.
            p_u = np.exp(self.alpha * (np.log(self.L0(u, t)[0])- C[u]))
            #print("p_u:", p_u)
            p_distr.append(p_u)

        p_sum = sum(p_distr)
        u_dic = {u: p/p_sum for u,p in zip(V,p_distr)}
        #print(p_distr)
        #print(u_dic)
        return(u_dic)

    def choose_u(self): 
        P_u = self.S1(self.target)
        mv = max(P_u.values())
        res = [k for k in P_u if P_u[k] == mv]
        return(rd.choice(res))
    

    def L1(self, u):
        t_distr = []
        for t in V: 
            p = self.S1(t)[u] * self.P_t[t]
            t_distr.append(p)

        p_sum = sum(t_distr)
        t_dic = {t: p/p_sum for t,p in zip(V,t_distr)}

        # is the new probability distribution a mix of the old distribution and the newly calculated one or is it just the new one? 
        self.P_t = t_dic

        return(t_dic)
    



class GameState:
    def __init__(self): 
        self.previous_words = []

    def say_word(self, word):
        self.previous_words += word